## NB-08 — Complete Solution

This is the capstone notebook. It assembles every component from NB-01 through
NB-07 into a single end-to-end CatalogueIQ pipeline, runs the RAGAS evaluation
suite against the golden test set from NB-02, prints a results table broken down
by query type, makes one targeted improvement, re-evaluates, and records the
delta. This notebook is the foundation for your demo and evaluation report.

### Concepts Covered

- Assembling the full CatalogueIQ pipeline from all component notebooks
- Running RAGAS evaluation against the 10-question golden test set
- Reading the results table: which query types score lowest and why
- The expected pattern: comparative recommendation and multi-hop score lowest
- Targeted improvement: increasing `top_k` from 3 to 6 for comparative recommendation queries
- Measuring and recording the RAGAS delta before and after improvement
- Saving results to `output/ragas_results.json` for the evaluation report

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────
import os
import json
import pickle
import time
from pathlib import Path
from dotenv import load_dotenv
import faiss
import numpy as np
from langchain.schema import Document
from langchain_anthropic import ChatAnthropic
from langchain.chains import RetrievalQA
from langchain.vectorstores import FAISS as LangchainFAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer, CrossEncoder
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

load_dotenv()

MODEL = "claude-sonnet-4-6"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
INDEX_DIR = Path("data/index")
os.makedirs("output", exist_ok=True)

assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found in .env"
print(f"Model: {MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")


In [ ]:
# ── LOAD ALL COMPONENTS ───────────────────────────────────────

# Load FAISS index
index_path = INDEX_DIR / "faiss.index"
meta_path  = INDEX_DIR / "metadata.pkl"
if not index_path.exists():
    raise FileNotFoundError("Run NB-04 to build the FAISS index first.")

with open(meta_path, "rb") as f:
    all_docs = pickle.load(f)

hf_embedder = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = LangchainFAISS.from_documents(all_docs, hf_embedder)

# Load golden test set from NB-02
golden_path = Path("output/golden_test_set.json")
if not golden_path.exists():
    raise FileNotFoundError("Run NB-02 to create golden_test_set.json first.")
with open(golden_path, encoding="utf-8") as f:
    golden_test_set = json.load(f)

print(f"Loaded {len(all_docs)} documents")
print(f"Loaded {len(golden_test_set)} golden test questions")

# LLM
llm = ChatAnthropic(model=MODEL, temperature=0, max_tokens=1024)


### Step 1 — Full Pipeline: System Prompt and QA Chain

We assemble the complete production pipeline with the groundedness-enforcing
system prompt from NB-05.

In [ ]:
# ── PRODUCTION QA CHAIN (baseline: top_k=3) ─────────────────

CATALOGUE_QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are CatalogueIQ, the product intelligence assistant for
ShopSmart India. Answer accurately using ONLY the retrieved context below.

RULES:
1. NEVER invent product specifications. If not in context, say so.
2. NEVER invent policy rules. Cite the exact section name.
3. If a product is not in context, state: "This product is not in the ShopSmart India catalogue."
4. Include at least one citation per factual claim: [Source: <file>, Product ID: <id> or Section: <heading>]
5. For comparative queries, compare ALL products found in context.
6. Use ₹ for Indian Rupee amounts.

CONTEXT:
{context}

QUESTION:
{question}

Answer:"""
)

def build_qa_chain(top_k: int = 3) -> RetrievalQA:
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": top_k},
    )
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": CATALOGUE_QA_PROMPT},
    )

baseline_chain = build_qa_chain(top_k=3)
print("Baseline QA chain built (top_k=3)")


### Step 2 — Generate Answers for the Full Golden Test Set

We run all 10 golden test questions through the baseline pipeline and collect
answers and contexts. This is the input to RAGAS evaluation.

In [ ]:
# COST NOTE: 10 questions × 1 API call each = 10 Claude API calls ≈ $0.05–$0.10

print("Generating answers for all 10 golden test questions...")
print("(This makes 10 Claude API calls — estimated cost < $0.10)")

baseline_answers = []
baseline_contexts = []

for i, item in enumerate(golden_test_set, 1):
    question = item["question"]
    print(f"  [{i:2d}/10] {item['query_type']}: {question[:55]}…")

    # COST NOTE: API call to Claude
    result = baseline_chain.invoke({"query": question})

    baseline_answers.append(result["result"])
    # RAGAS expects contexts as list[str] per question
    contexts = [doc.page_content for doc in result.get("source_documents", [])]
    baseline_contexts.append(contexts)

print(f"\n✅ Generated {len(baseline_answers)} answers.")


### Step 3 — RAGAS Baseline Evaluation

We run the four RAGAS metrics against all 10 questions and print results broken
down by query type. The expected pattern: comparative recommendation and multi-hop
queries score lowest because they require multiple chunks and cross-document reasoning.

In [ ]:
# COST NOTE: RAGAS makes ~3-5 API calls per question per metric.
# 10 questions × 4 metrics × 3 calls ≈ 120 Claude API calls ≈ $0.20–$0.40.

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(hf_embedder)

baseline_dataset = Dataset.from_dict({
    "question":     [item["question"] for item in golden_test_set],
    "answer":       baseline_answers,
    "contexts":     baseline_contexts,
    "ground_truth": [item["ground_truth"] for item in golden_test_set],
})

print("Running RAGAS baseline evaluation (this takes 2-5 minutes)...")
print("Metrics: faithfulness, answer_relevancy, context_precision, context_recall")

# COST NOTE: ~120 API calls to Claude for the full evaluation
baseline_ragas = evaluate(
    dataset=baseline_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

baseline_df = baseline_ragas.to_pandas()
baseline_df["query_type"] = [item["query_type"] for item in golden_test_set]

print("\n── RAGAS Baseline Results ───────────────────────────────────")
print(f"{'Query Type':<30} {'Faith':>6} {'AnsRel':>7} {'CtxPrec':>8} {'CtxRec':>7}")
print("─" * 60)

for qt in ["product_factual", "policy_eligibility", "comparative_recommendation",
           "seller_policy", "multi_hop"]:
    subset = baseline_df[baseline_df["query_type"] == qt]
    if len(subset) == 0:
        continue
    faith   = subset["faithfulness"].mean()
    ansrel  = subset["answer_relevancy"].mean()
    ctxprec = subset["context_precision"].mean()
    ctxrec  = subset["context_recall"].mean()
    print(f"{qt:<30} {faith:>6.3f} {ansrel:>7.3f} {ctxprec:>8.3f} {ctxrec:>7.3f}")

print("─" * 60)
overall = baseline_df[["faithfulness","answer_relevancy","context_precision","context_recall"]].mean()
print(f"{'OVERALL MEAN':<30} {overall['faithfulness']:>6.3f} {overall['answer_relevancy']:>7.3f} "
      f"{overall['context_precision']:>8.3f} {overall['context_recall']:>7.3f}")
print("\n⚠️  Expected: comparative_recommendation and multi_hop score lowest.")


### Step 4 — Targeted Improvement: Increase top_k for Comparative Queries

The analysis shows comparative recommendation and multi-hop queries score lowest.
The most direct fix for comparative: increase `top_k` from 3 to 6, so more
product chunks are retrieved and compared. We re-run only the comparative and
multi-hop questions with the improved chain.

In [ ]:
# COST NOTE: 4 questions × 1 API call + RAGAS ≈ $0.10 additional

print("Applying targeted improvement: top_k=3 → top_k=6 for comparative + multi-hop queries")
improved_chain = build_qa_chain(top_k=6)

# Re-generate answers for comparative and multi-hop questions only
improved_answers = list(baseline_answers)    # copy baseline
improved_contexts = list(baseline_contexts)

target_types = {"comparative_recommendation", "multi_hop"}

for i, item in enumerate(golden_test_set):
    if item["query_type"] in target_types:
        print(f"  Re-running [{i+1}]: {item['query_type']}: {item['question'][:50]}…")
        # COST NOTE: API call to Claude
        result = improved_chain.invoke({"query": item["question"]})
        improved_answers[i] = result["result"]
        improved_contexts[i] = [doc.page_content for doc in result.get("source_documents", [])]

print("\nRe-running RAGAS for improved answers...")

improved_dataset = Dataset.from_dict({
    "question":     [item["question"] for item in golden_test_set],
    "answer":       improved_answers,
    "contexts":     improved_contexts,
    "ground_truth": [item["ground_truth"] for item in golden_test_set],
})

# COST NOTE: Another ~120 RAGAS API calls
improved_ragas = evaluate(
    dataset=improved_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

improved_df = improved_ragas.to_pandas()
improved_df["query_type"] = [item["query_type"] for item in golden_test_set]

print("\n── RAGAS Results: Baseline vs Improved (top_k 3→6) ─────────")
print(f"{'Query Type':<30} {'Faith Δ':>8} {'AnsRel Δ':>9} {'CtxPrec Δ':>10} {'CtxRec Δ':>9}")
print("─" * 70)

for qt in ["product_factual", "policy_eligibility", "comparative_recommendation",
           "seller_policy", "multi_hop"]:
    b_sub = baseline_df[baseline_df["query_type"] == qt]
    i_sub = improved_df[improved_df["query_type"] == qt]
    if len(b_sub) == 0:
        continue
    for metric in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
        b_mean = b_sub[metric].mean()
        i_mean = i_sub[metric].mean()

    # Print delta row
    delta_faith   = i_sub["faithfulness"].mean()       - b_sub["faithfulness"].mean()
    delta_ansrel  = i_sub["answer_relevancy"].mean()   - b_sub["answer_relevancy"].mean()
    delta_ctxprec = i_sub["context_precision"].mean()  - b_sub["context_precision"].mean()
    delta_ctxrec  = i_sub["context_recall"].mean()     - b_sub["context_recall"].mean()

    arrow = lambda d: "▲" if d > 0.01 else ("▼" if d < -0.01 else "→")
    print(f"{qt:<30} "
          f"{arrow(delta_faith)}{delta_faith:+.3f}  "
          f"{arrow(delta_ansrel)}{delta_ansrel:+.3f}   "
          f"{arrow(delta_ctxprec)}{delta_ctxprec:+.3f}    "
          f"{arrow(delta_ctxrec)}{delta_ctxrec:+.3f}")


In [ ]:
# ── SAVE ALL RESULTS ─────────────────────────────────────────

results_output = {
    "metadata": {
        "model": MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "baseline_top_k": 3,
        "improved_top_k": 6,
        "total_questions": len(golden_test_set),
        "query_types": ["product_factual", "policy_eligibility",
                        "comparative_recommendation", "seller_policy", "multi_hop"],
    },
    "baseline": {
        "overall": {
            "faithfulness":      float(baseline_df["faithfulness"].mean()),
            "answer_relevancy":  float(baseline_df["answer_relevancy"].mean()),
            "context_precision": float(baseline_df["context_precision"].mean()),
            "context_recall":    float(baseline_df["context_recall"].mean()),
        },
        "by_query_type": {},
        "per_question": [],
    },
    "improved": {
        "overall": {
            "faithfulness":      float(improved_df["faithfulness"].mean()),
            "answer_relevancy":  float(improved_df["answer_relevancy"].mean()),
            "context_precision": float(improved_df["context_precision"].mean()),
            "context_recall":    float(improved_df["context_recall"].mean()),
        },
        "by_query_type": {},
        "per_question": [],
    },
}

# By query type breakdowns
for qt in ["product_factual", "policy_eligibility", "comparative_recommendation",
           "seller_policy", "multi_hop"]:
    for label, df in [("baseline", baseline_df), ("improved", improved_df)]:
        sub = df[df["query_type"] == qt]
        if len(sub):
            results_output[label]["by_query_type"][qt] = {
                "faithfulness":      float(sub["faithfulness"].mean()),
                "answer_relevancy":  float(sub["answer_relevancy"].mean()),
                "context_precision": float(sub["context_precision"].mean()),
                "context_recall":    float(sub["context_recall"].mean()),
            }

# Per-question detail
for i, (item, b_ans, i_ans) in enumerate(
    zip(golden_test_set, baseline_answers, improved_answers)
):
    b_row = baseline_df.iloc[i]
    i_row = improved_df.iloc[i]
    results_output["baseline"]["per_question"].append({
        "question": item["question"],
        "query_type": item["query_type"],
        "answer": b_ans,
        "faithfulness":      float(b_row["faithfulness"]),
        "answer_relevancy":  float(b_row["answer_relevancy"]),
        "context_precision": float(b_row["context_precision"]),
        "context_recall":    float(b_row["context_recall"]),
    })
    results_output["improved"]["per_question"].append({
        "question": item["question"],
        "query_type": item["query_type"],
        "answer": i_ans,
        "faithfulness":      float(i_row["faithfulness"]),
        "answer_relevancy":  float(i_row["answer_relevancy"]),
        "context_precision": float(i_row["context_precision"]),
        "context_recall":    float(i_row["context_recall"]),
    })

out_path = "output/ragas_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results_output, f, indent=2, ensure_ascii=False)

print(f"\n✅ Full RAGAS results saved to {out_path}")
print(f"\nBaseline overall: faithfulness={results_output['baseline']['overall']['faithfulness']:.3f}, "
      f"answer_relevancy={results_output['baseline']['overall']['answer_relevancy']:.3f}")
print(f"Improved overall:  faithfulness={results_output['improved']['overall']['faithfulness']:.3f}, "
      f"answer_relevancy={results_output['improved']['overall']['answer_relevancy']:.3f}")


In [ ]:
# ── EXPERIMENT ────────────────────────────────────────────────

# EXERCISE 1 — Second improvement: query expansion
# Implement multi_query_retrieve from NB-06 in the baseline chain.
# Re-run RAGAS for comparative and multi-hop questions.
# Does expansion + top_k=6 outperform top_k=6 alone?
# Which metric improves most: context_precision or context_recall?

# EXERCISE 2 — Analyse the lowest-scoring question
# Find the question with the lowest faithfulness score in baseline_df.
# Print the full answer and the retrieved contexts for that question.
# Identify: is the model hallucinating, or is the context genuinely missing
# the information needed? What would fix it?

# EXERCISE 3 — Simulate a 100k product catalogue
# Add 100 duplicate product documents to all_docs with random price_inr values.
# Rebuild the vectorstore and re-run retrieval for the comparative query.
# Does context_precision drop? Does response time increase?
# What would you change first if ShopSmart India added 100,000 new products?

print("Final experiments ready. Modify chains and re-run RAGAS to measure improvements.")


### Key Takeaway

The complete CatalogueIQ pipeline demonstrates the full RAG engineering cycle:
define quality metrics first (NB-02), build and evaluate (NB-08), identify the
lowest-scoring query type, make one targeted change, and measure the delta.
The pattern — comparative recommendation and multi-hop queries score lowest,
and increasing `top_k` improves `context_recall` for those types — is
predictable and now you can explain *why* to the panel. The `output/ragas_results.json`
file is your evidence: every design decision in the evaluation report should
connect back to a row in that table.